# 04 — Weapon Comparison

Compare two weapons on the same character against the same NPC target.

Edit the **Configuration** cell to change:
- Attacker class, stats, skill level
- The two weapons to compare
- The NPC target stats and armor

In [ ]:
import logging
from pathlib import Path

from omega.model.constants import (
    SKILLID_ANATOMY, SKILLID_ENTICEMENT, SKILLID_FENCING,
    SKILLID_MUSICIANSHIP, SKILLID_PEACEMAKING, SKILLID_PROVOCATION,
    SKILLID_SWORDSMANSHIP, SKILLID_TACTICS, SKILLID_WRESTLING,
)
from omega.shard import ShardData
from omega.simulation import (
    ArmorSpec, CombatantSpec, Scenario, WeaponSpec, run_scenario,
)
from omega.reporting.tables import comparison_table, format_table_html
from omega.reporting.plots import comparison_breakdown, comparison_overlay
from omega.logging import setup_logging

# Suppress noisy stub warnings — only show errors in notebook output
setup_logging(level=logging.ERROR)

SHARD_ROOT = Path("submodules/zuluhotel_omega_2.5")
if not SHARD_ROOT.exists():
    SHARD_ROOT = Path("../submodules/zuluhotel_omega_2.5")
shard = ShardData.from_path(SHARD_ROOT)

## Configuration

In [ ]:
# --- Attacker: Bladesinger with all in-class skills ---
SKILL_LEVEL = 100
BLADESINGER_SKILLS = {
    SKILLID_ANATOMY: SKILL_LEVEL,
    SKILLID_ENTICEMENT: SKILL_LEVEL,
    SKILLID_FENCING: SKILL_LEVEL,
    SKILLID_MUSICIANSHIP: SKILL_LEVEL,
    SKILLID_PEACEMAKING: SKILL_LEVEL,
    SKILLID_PROVOCATION: SKILL_LEVEL,
    SKILLID_SWORDSMANSHIP: SKILL_LEVEL,
    SKILLID_TACTICS: SKILL_LEVEL,
}

ATTACKER_STATS = {"str": 100, "dex": 100, "int": 25}
CLASS_LEVEL = 5

# --- Two weapons to compare ---
WEAPON_A = WeaponSpec(name="Broadsword", damage="3d6+2")
WEAPON_B = WeaponSpec(name="Katana", damage="2d8+4")

# --- NPC target ---
DEFENDER = CombatantSpec(
    name="Target", is_npc=True,
    str_=50, dex_=50, int_=50, hp=500,
    skills={SKILLID_WRESTLING: 70},  # NPC combat skill — affects hit chance
    armor=ArmorSpec(ar=30),
)

# --- Simulation settings ---
ITERATIONS = 200
BASE_SEED = 42

print(f"Bladesinger (level {CLASS_LEVEL}, skills @ {SKILL_LEVEL})")
print(f"  STR={ATTACKER_STATS['str']}  DEX={ATTACKER_STATS['dex']}  INT={ATTACKER_STATS['int']}")
print(f"Weapon A: {WEAPON_A.name} ({WEAPON_A.damage})")
print(f"Weapon B: {WEAPON_B.name} ({WEAPON_B.damage})")
print(f"Target: AR {DEFENDER.armor.ar}, HP {DEFENDER.hp}, Wrestling {DEFENDER.skills.get(SKILLID_WRESTLING, 0)}")

## Run Simulation

In [ ]:
RUN_KW = dict(shard=shard)

def make_attacker(weapon):
    return CombatantSpec(
        name=f"Bladesinger ({weapon.name})",
        skills=BLADESINGER_SKILLS,
        str_=ATTACKER_STATS["str"],
        dex_=ATTACKER_STATS["dex"],
        int_=ATTACKER_STATS["int"],
        class_levels={"IsBladesinger": CLASS_LEVEL},
        weapon=weapon,
    )

result_a = run_scenario(
    Scenario(attacker=make_attacker(WEAPON_A), defender=DEFENDER,
             iterations=ITERATIONS, base_seed=BASE_SEED),
    **RUN_KW,
)
result_b = run_scenario(
    Scenario(attacker=make_attacker(WEAPON_B), defender=DEFENDER,
             iterations=ITERATIONS, base_seed=BASE_SEED),
    **RUN_KW,
)

results = {WEAPON_A.name: result_a, WEAPON_B.name: result_b}

for label, r in results.items():
    ds = r.damage_stats
    ds_hit = r.damage_stats_on_hit
    print(f"  {label:20s}  mean={ds.mean:6.2f}  on_hit={ds_hit.mean:6.2f}  hit_rate={r.ratios.hit_rate:.1%}")

In [ ]:
# Overlaid damage distributions
comparison_overlay(
    results,
    title=f"Weapon Comparison — {WEAPON_A.name} vs {WEAPON_B.name}",
)

In [ ]:
# Stacked bar: base damage, absorbed, final
comparison_breakdown(
    results,
    title=f"Damage Breakdown — {WEAPON_A.name} vs {WEAPON_B.name}",
)

In [ ]:
# Side-by-side comparison table
from IPython.display import HTML

rows = comparison_table(
    results,
    stats=["mean", "mean_on_hit", "median", "min", "max", "p5", "p95",
           "hit_rate", "absorbed_mean"],
)
HTML(format_table_html(rows))

## Enchanted & Elemental Weapons (V1.5)

Compare the same base weapon with different V1.5 modifications:

- **Daemon's Breath** — spell strike enchantment (casts Fireball on 75% of hits)
- **Slayer** — slayer enchantment (2x damage multiplier vs Undead targets, tested against Undead)
- **Vampiric** — effect enchantment (drains defender mana on every hit)
- **Fire Elemental** — inline elemental split (50% fire / 50% physical, no hitscript — elemental
  damage is calculated in the main damage formula, hence `elem_total_net > 0` but
  `spell_strike_rate = 0`)

In [ ]:
import dataclasses
from omega.config.enchantments import Enchantment
from omega.reporting.plots import enchantment_comparison

base_sword = WeaponSpec(name="Broadsword", damage="3d6+2")

# Spell strike weapons need ChanceOfEffect and EffectCircle to actually fire
spell_base = WeaponSpec(
    name="Broadsword", damage="3d6+2",
    properties={"ChanceOfEffect": 75, "EffectCircle": 10},
)

enchanted_weapons = {
    "Plain": base_sword,
    "Daemon's Breath (75%)": spell_base.enchant_with(Enchantment.OF_DAEMONS_BREATH),
    "Slayer (vs Undead)": base_sword.enchant_with(Enchantment.SILVER),
    "Vampiric (mana drain)": base_sword.enchant_with(Enchantment.VAMPIRIC),
    "Fire Elemental (50/50)": WeaponSpec(
        name="Fire Broadsword", damage="3d6+2",
        properties={"ElementalDamage": "FIRE:50 PHYSICAL:50"},
    ),
}

# Slayer needs an Undead target to show the damage bonus
UNDEAD_DEFENDER = dataclasses.replace(DEFENDER, name="Skeleton", properties={"Type": "Undead"})

enchant_results = {}
for label, weapon in enchanted_weapons.items():
    # Use Undead target for Slayer to show slayer bonus
    defender = UNDEAD_DEFENDER if "Slayer" in label else DEFENDER
    enchant_results[label] = run_scenario(
        Scenario(attacker=make_attacker(weapon), defender=defender,
                 iterations=ITERATIONS, base_seed=BASE_SEED),
        **RUN_KW,
    )
    ds = enchant_results[label].damage_stats
    r = enchant_results[label].ratios
    rates = [f"hit={r.hit_rate:.0%}"]
    if r.spell_strike_rate > 0:
        rates.append(f"spell_strike={r.spell_strike_rate:.0%} (on hit: {r.spell_strike_rate_on_hit:.0%})")
    if r.effect_rate > 0:
        rates.append(f"effect={r.effect_rate:.0%} (on hit: {r.effect_rate_on_hit:.0%})")
    rate_str = f"  ({', '.join(rates)})"
    print(f"  {label:28s}  mean={ds.mean:6.2f}{rate_str}")

In [ ]:
# Bar chart comparing mean damage across enchantments
enchantment_comparison(enchant_results, title="Enchantment Effectiveness")

In [ ]:
# Full comparison table with V1.5 stat columns
rows = comparison_table(
    enchant_results,
    stats=["mean", "mean_on_hit", "median", "p5", "p95", "hit_rate",
           "spell_strike_rate", "spell_strike_rate_on_hit",
           "effect_rate", "effect_rate_on_hit",
           "drain_mean", "drain_mean_on_hit", "elem_total_net"],
)
HTML(format_table_html(rows))